# Strategic Analysis of the AI/ML Thesis Landscape at VŠE

This report applies Text Analytics techniques to a curated corpus of bachelor theses from the VŠKP catalog.

## Introduction & Research Methodology

The findings are structured to guide the Vice-Dean for Research across three core operational decisions:
* **Curriculum Development:** Identifying which AI/ML sub-fields have strong student adoption.
* **Student Recruitment & Marketing:** Defining how FIS can credibly position its technical strengths to applicants.
* **Industry Partnerships:** Spotting real-world application domains that appear most frequently in student research to target internships and agreements.

### Data Corpus and Multi-Dimensional Schema
The research corpus consists of **55 unique bachelor theses** extracted via the VŠKP portal. To build a dataset that captures targeted trends in data engineering and computational linguistics, the retrieval queried four specific thematic branches within the text fields:
1. `"natural language processing"`
2. `"sentiment analysis"`
3. `"transformer"`
4. `"large language model"`

To prevent an uneven distribution from a single massive topic, I've capped each individual query at a maximum of 15 unique links during retrieval, yielding a highly concentrated and clean baseline of 55 theses.

#### Overcoming Abstract Text Limitations
This project engineers a multi-dimensional schema that references text with concrete metadata fields:
* **Textual Layers:** Title, Abstract, and Keywords.
* **Institutional Context:** Study Program, Faculty, and Academic Year.
* **Academic Layer:** Assigned Supervisor  and Opponents.

### Language and Boundary Constraints
Natural Language Processing tools, tokenizers, and deep sentence embedding architectures achieve significantly higher semantic accuracy on English text due to broader pre-training distributions. Mixing languages would introduce severe semantic misalignment into the vector space, creating artificial clusters based on language signatures rather than research content.

Localized Czech academic terminology (e.g., specific study program like *Aplikovaná informatika*) is preserved exactly as scraped. These categorical parameters are used exclusively for rule-based matching and institutional grouping, with English translations added into the final visualizations where necessary for clarity.

### Theoretical Machine Learning Framework
To map out the thematic space, the methodology skips classical count-based sparse models such as TF-IDF in favor of deep dense vector representations for clustering. TF-IDF is reserved for downstream cluster labeling, where sparse term distinctiveness is the goal. We employ the pre-trained `all-MiniLM-L6-v2` SentenceTransformer model, which embeds text sequences into a 384-dimensional dense space where semantic proximity translates directly to cosine similarity. This enables the model to map synonyms (e.g., identifying that "deep neural network" and "CNN" belong to overlapping conceptual spaces) that traditional keyword matchers would miss.

The thematic partitioning is resolved using **K-means Clustering** optimized on normalized embeddings. Given our localized dataset of 55 theses, density-based alternatives like HDBSCAN are inappropriate because they discard unaligned data points as noise. K-means ensures a clean, exhaustive partition where every thesis is accounted for.

In [ ]:
!pip install -q sentence-transformers umap-learn scikit-learn plotly langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 7.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
import requests, time, re, html, os, umap
import pandas as pd, numpy as np, networkx as nx

from bs4 import BeautifulSoup
from urllib.parse import urlencode

import plotly.express as px
import plotly.graph_objects as go

from collections import Counter, defaultdict
from sentence_transformers import SentenceTransformer

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.preprocessing import normalize
from sklearn.feature_extraction.text import TfidfVectorizer

from networkx.algorithms.community import greedy_modularity_communities
from langdetect import detect, LangDetectException

## Data retrieval
I extracted the dataset directly from the VŠKP portal using its English-language interface `https://vskp.vse.cz/english`. I chose this approach to standardize the underlying HTML structure, ensuring that the metadata tables, field labels, and language tags were captured using a uniform schema. This choice does not imply that the theses in the corpus were written in English; rather, it ensures that the scraped metadata fields are structurally consistent across the entire dataset.

To capture the specific landscape of text analytics, data engineering, and modern linguistic models, I targeted the abstract fields using four distinct searches:
1. `"natural language processing"`
2. `"sentiment analysis"`
3. `"transformer"`
4. `"large language model"`

To prevent a single massive topic from overwhelming the dataset and distorting the downstream distributions, I capped the results at a maximum of 15 unique links per query. In the end I've gotten corpus of unique 55 theses. To be kind to the portal's usage, a 1.5-second request delay between individual thesis pages and a 2-second pause between query executions was used.

### 2.2 Corpus Schema Definition
To address the weakness of relying solely on abstracts, I extracted a comprehensive multi-layered schema for each thesis:
* **Textual Elements:** `title`, `abstract`, and `keywords`
* **Institutional Context:** `program`, `faculty`, and `department`
* **Temporal Tracking:** `date_assigned`, `date_submitted`, and `date_defended`
* **Academic Network:** `author`, `supervisor`, and `opponents`

### Scraping

If you already posses the `csv` file, upload the file to the notebook and load the data to skip the scraping process.

In [ ]:
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "Mozilla/5.0 (academic research scraper)"})
BASE_URL = "https://vskp.vse.cz"
ABSTRACT_QUERIES = [
    "natural language processing",
    "sentiment analysis",
    "transformer",
    "large language model"
]

def build_search_url(query, page=1):
    params = {
        "type": "Bakalářská práce",
        "abstract": query,
        "page": page,
    }
    # Must use urlencode, not raw f-string with dict
    return f"{BASE_URL}/english?{urlencode(params, doseq=True)}"


def get_thesis_links(query, max_pages=3, max_results=15):
    links = []
    for page in range(1, max_pages + 1):
        url = build_search_url(query, page)
        resp = SESSION.get(url, timeout=10)
        if resp.status_code != 200:
            print(f"{url} -> {resp.status_code}")
            break
        soup = BeautifulSoup(resp.text, "html.parser")

        page_links = list({
            re.sub(r'\?.*$', '', a["href"])
            for a in soup.find_all("a", href=re.compile(r'^/english/\d+'))
        })
        if not page_links:
            print(f"  No links found on page {page}")
            break

        links.extend(page_links)
        print(f"Query '{query}' page {page}: {len(page_links)} links")

        if len(links) >= max_results:
            break # stop paginating
        time.sleep(1.5)

    return list(set(links))[:max_results] # dedup and cap


def scrape_thesis(path):
    if path.startswith("/english"):
        url = BASE_URL + path
    else:
        url = f"{BASE_URL}/english{path}"

    resp = SESSION.get(url, timeout=10)
    if resp.status_code != 200:
        return None

    soup = BeautifulSoup(resp.text, "html.parser")

    # English tab pane: title, author, type, language, abstract, keywords
    en_pane = soup.find("div", id="thesisDetails-en")
    if not en_pane:
        return None

    en_details = {}
    for table in en_pane.find_all("table", class_="thesis-details-table"):
        for row in table.find_all("tr"):
            th = row.find("th")
            td = row.find("td")
            if not th or not td:
                continue
            label = th.get_text(strip=True).lower().replace(":", "").replace(" ", "_")
            en_details[label] = td.get_text(" ", strip=True)

    # Global tables: study programme + defense dates
    global_details = {}
    tab_content = soup.find("div", id="thesisDetailsContent")
    if tab_content:
        for table in tab_content.find_all("table", class_="thesis-details-table", recursive=False):
            for row in table.find_all("tr"):
                th = row.find("th")
                td = row.find("td")
                if not th or not td:
                    continue
                label = th.get_text(strip=True).lower().replace(":", "").replace(" ", "_")
                global_details[label] = td.get_text(" ", strip=True)

    # en_details overrides global_details on any key collision
    details = {**global_details, **en_details}

    # Abstract from EN pane
    abstract_div = en_pane.find("div", class_="abstract")
    abstract = abstract_div.get_text(" ", strip=True) if abstract_div else en_details.get("abstract")

    # h1 fallback if thesis_title somehow missing
    if not details.get("thesis_title"):
        h1 = soup.find("h1")
        details["thesis_title"] = h1.get_text(strip=True) if h1 else None

    return {
        "url":           url,
        "title":         details.get("thesis_title"),
        "abstract":      abstract,
        "keywords":      details.get("keywords"),
        "author":        details.get("author"),
        "supervisor":    details.get("supervisor"),
        "opponents":     details.get("opponents"),
        "thesis_type":   details.get("thesis_type"),
        "language":      details.get("thesis_language"),
        "program":       details.get("study_programme"),
        "program_type":  details.get("type_of_study_programme"),
        "degree":        details.get("assigned_degree"),
        "institution":   details.get("institutions_assigning_academic_degree"),
        "faculty":       details.get("faculty"),
        "department":    details.get("department"),
        "date_assigned": details.get("date_of_assignment"),
        "date_submitted":details.get("date_of_submission"),
        "date_defended": details.get("date_of_defense"),
        "insis_url":     details.get("identifier_in_the_insis_system"),
    }

# Collection loop
all_links = []
for q in ABSTRACT_QUERIES:
    print(f"\nSearching: '{q}'")
    links = get_thesis_links(q, max_pages=1)
    print(links)
    all_links.extend(links)
    time.sleep(2)

all_links = list(set(all_links))
print(f"\nTotal unique thesis pages (With each topic capped at 15): {len(all_links)}")

records = []
for i, path in enumerate(all_links):
    record = scrape_thesis(path)
    if record and record.get("abstract"):
        records.append(record)
    if i % 10 == 0:
        print(f"Scraped {i}/{len(all_links)}")
    time.sleep(1.5)

df_raw = pd.DataFrame(records).drop_duplicates(subset="url")
df_raw.to_csv("theses_raw.csv", index=False)
print(f"\nSaved {len(df_raw)} theses to theses_raw.csv")


Searching: 'natural language processing'
Query 'natural language processing' page 1: 18 links
['/english/90410_comparison-of-communication-style-of-fortune-500-ceos-by-gender', '/english/83435_implementation-of-a-chatbot-for-the-organization-junak-cesky-skaut-z-s', '/english/83432_comparison-of-cloud-ml-platforms-with-a-focus-on-natural-language-processing', '/english/96587_analysis-of-attitudes-and-opinions-of-users-of-social-networks-regarding-super-apps', '/english/97179_innovation-in-insurance-through-ai', '/english/92839_comparing-large-language-models-analysis-of-functionality-and-limitations', '/english/96859_detecting-malicious-content-on-the-internet-using-machine-learning', '/english/96892_data-analysis-of-vocabulary-and-sentiment-in-english-literature-as-a-tool-for-foreign-language-learning', '/english/76741_the-most-significant-corpora-in-czech-and-english-for-natural-language-processing', '/english/42100_sentiment-analysis-in-the-czech-environment-network-twitter', '/engl

### Loading scraped data

In [ ]:
def load_theses(filepath="theses_raw.csv"):
    """Load scraped thesis data from CSV."""
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"{filepath} not found. Run the scraper first.")

    df = pd.read_csv(filepath, encoding="utf-8")

    # Sanity checks
    required = ["url", "title", "abstract"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"CSV missing columns: {missing}")

    # Drop rows with empty abstracts
    df = df.dropna(subset=["abstract"]).reset_index(drop=True)

    print(f"Loaded {len(df)} theses from {filepath}")
    return df


def load_theses_records(filepath="theses_raw.csv"):
    """Load as list of dicts."""
    df = load_theses(filepath)
    return df.to_dict("records")

df_raw = load_theses()

# Quick overview
print(df_raw.head(5)[["title", "author", "date_submitted"]])
print(f"\nAbstract lengths — mean: {df_raw['abstract'].str.len().mean():.0f}, "
      f"min: {df_raw['abstract'].str.len().min()}, max: {df_raw['abstract'].str.len().max()}")

Loaded 55 theses from theses_raw.csv
                                               title            author  \
0  Implementation of a chatbot for the organizati...     Medek, Štěpán   
1  AI Misalignment: Undesirable and Misaligned Be...      Holan, Lukáš   
2  Image classification using machine learning me...  Tatarinov, Vadym   
3  Prediction and Analysis of Vehicle Safety Ratings     Havlík, Roman   
4  Potential Applications of GPT-4o for Cyber Thr...    Levý, Jindřich   

  date_submitted  
0    10. 5. 2021  
1     5. 5. 2026  
2    8. 12. 2024  
3    11. 5. 2026  
4    12. 5. 2025  

Abstract lengths — mean: 1367, min: 397, max: 2732


## Data cleaning
Before running any text embedding or categorization models, I implemented a data cleaning pipeline to fix structural problems in the raw scraped data:

1. **HTML Entity Resolution:** Text fields contained encoded web characters (e.g., `&apos;`, `&amp;`). I forced an unescaping pass using `html.unescape()` across all textual columns (`abstract`, `title`, `keywords`) to restore accurate punctuation.
2. **Temporal Parameter Extraction:** The submission dates were trapped inside full raw strings (e.g., `"20. 4. 2026"`). I isolated academic year using regex tokenization (`\b(20\d{2})\b`).
3. **Categorical Normalization:** Study program tags contained variations due to curriculum updates over time (e.g., tracking both `"Aplikovaná informatika"` and `"Aplikovaná informatika/Aplikovaná informatika"`). I've merged and translated them accordingly.

### Language Filtering
Because text analytics models achieve maximum alignment when operating on a single language space, I verified the language distribution of the corpus. I have used `langdetect` to confirm consistent language across abstracts.

In [ ]:
df = df_raw.copy()

# Unescape HTML entities
for col in ["abstract", "title", "keywords"]:
    df[col] = df[col].apply(lambda x: html.unescape(str(x)) if pd.notna(x) else x)

# Extract year from date_defended
def extract_year(date_str):
    match = re.search(r'\b(20\d{2})\b', str(date_str))
    return int(match.group(1)) if match else None

df["year"] = df["date_defended"].apply(extract_year)
print(f"Year range: {df['year'].min()}-{df['year'].max()}")
print(f"Missing year: {df['year'].isnull().sum()}")

# Language detection on abstracts
def detect_lang(text):
    try:
        return detect(str(text))
    except:
        return "unknown"

df["lang"] = df["abstract"].apply(detect_lang)
lang_counts = df["lang"].value_counts()
print(f"\nLanguage distribution:\n{lang_counts}")

n_dropped = (df["lang"] != "en").sum()
df = df[df["lang"] == "en"].reset_index(drop=True)
print(f"\nDropped {n_dropped} non-English abstracts")

# Program cleaning and merging
# We save original values to program_old and cleaned values to program
PROGRAM_MAP = {
    "Aplikovaná informatika": "Applied Informatics",
    "Aplikovaná informatika/Aplikovaná informatika": "Applied Informatics",
    "Aplikovaná informatika/Informatika": "Applied Informatics",
    "Data Analytics": "Data Analytics",
    "Podniková ekonomika a management": "Business Administration",
    "Podniková ekonomika a\xa0management": "Business Administration",
    "Informační média a služby": "Information Media",
    "Informační média a\xa0služby": "Information Media",
    "Management": "Management",
    "Ekonomika a management/Management": "Management",
    "Mezinárodní ekonomické vztahy/Mezinárodní obchod": "International Business & Trade",
    "International Business": "International Business & Trade"
}

df["program_old"] = df["program"]
df["program_temp"] = df["program"].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()

df["program"] = df["program_temp"].map(PROGRAM_MAP).fillna("Other Programs")

program_counts = df["program"].value_counts()
df.loc[df["program"].isin(program_counts[program_counts < 2].index), "program"] = "Other programs"

print(df["program"].value_counts())

# Summary
print(f"\nFinal corpus: {len(df)} theses")
print(f"Abstract length mean: {df['abstract'].str.len().mean():.0f}, "
      f"min: {df['abstract'].str.len().min()}, max: {df['abstract'].str.len().max()}")
print(f"\nFaculty breakdown:")
print(df["faculty"].value_counts())

df.to_csv("theses_clean.csv", index=False)
print("\nSaved to theses_clean.csv")

Year range: 2014-2026
Missing year: 0

Language distribution:
lang
en    55
Name: count, dtype: int64

Dropped 0 non-English abstracts
program
Applied Informatics               30
Data Analytics                    10
Management                         4
Business Administration            4
Other Programs                     3
International Business & Trade     3
Other programs                     1
Name: count, dtype: int64

Final corpus: 55 theses
Abstract length mean: 1363, min: 397, max: 2732

Faculty breakdown:
faculty
Faculty of Informatics and Statistics    42
Faculty of International Relations        5
Faculty of Management                     4
Faculty of Business Administration        4
Name: count, dtype: int64

Saved to theses_clean.csv


In [ ]:
# Quick overview
print(df.info())
print(df.head())

# Analytical views

The corpus consists of **55 bachelor theses** from VŠE, all with English abstracts, spanning 20014-2026. Each thesis is represented by its abstract — the canonical, author-written summary of the work.


## View: Thematic Map of AI/ML Topics

**Question:** What are the main thematic clusters of AI/ML work across VŠE bachelor theses, and what characterizes each?

**Relevance:** Foundational view. These cluster labels are reused in all later views to cross-tabulate study programs, technology usage, and temporal trends.

**Method:** Each thesis is represented by a concatenation of `title`, `abstract`, and `keywords` — students often name specific tools only in the keywords, so combining the three gives a richer signal than the abstract alone. Texts were encoded with `all-MiniLM-L6-v2` (384-dim dense vectors, cosine ≈ conceptual similarity), then partitioned with K-means on L2-normalized embeddings.

**On choosing k:** I did *not* pick k by silhouette. Across k=3 to k=10 the silhouette never exceeds diff of 0.06 and the values sit within noise of each other. The corpus was selected on four NLP/LLM-adjacent queries, so the theses share heavy common vocabulary and genuinely cluster tightly in embedding space; the low score is itself a finding about corpus homogeneity, not something to optimize away. I instead selected **k by interpretability and stability**:
- I inspected k=3, 4, 5. k=3 lumps chatbots, LLM governance, and misinformation into a single oversized 'generative AI' bucket; k=5 separates strategically distinct themes (chatbots vs. LLM governance vs. misinformation) that leadership would treat differently. I selected **k=5**.
- **Stability:** I re-ran K-means across 20 random seeds and measured pairwise Adjusted Rand Index against the base partition. Mean ARI ≈ **0.46** (min 0.34) — *moderate*, not high. The large, distinctive clusters (Sentiment Analysis; Conversational AI) are stable across seeds; the boundary between the three LLM-adjacent clusters (governance, misinformation, applied) is softer and some theses migrate between them. This is consistent with a topically homogeneous corpus where themes overlap rather than sit in discrete groups. I therefore read the clusters as **an indicative thematic structure, not hard partitions** — the labels describe centres of gravity, and borderline theses legitimately belong to more than one.

K-means was kept over HDBSCAN because HDBSCAN discards low-density points as noise; with only 55 theses every document must belong to a cluster for the downstream cross-tabs. Clusters are characterized with **c-TF-IDF** (TF-IDF over each cluster's pooled text) for distinctive terms, cross-checked against the 5 most central titles per centroid. UMAP (n_neighbors=15, min_dist=0.1) is used only for the 2D map

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Combine textual columns dynamically in memory to enrich the semantic payload
print("Compiling and embedding multi-layered text features (Title + Abstract + Keywords)")
combined_text_series = df.apply(
    lambda row: f"{row['title']} {row['abstract']} {row['keywords']}",
    axis=1
).tolist()

raw_embeddings = embedding_model.encode(combined_text_series, show_progress_bar=False)
normalized_embeddings = normalize(raw_embeddings)

# Determine optimal partitioning threshold via Silhouette score optimization
silhouette_metrics = {}
print("Evaluating cluster separation quality across parametric range (k=3 to k=10)")
for k_candidate in range(3, 11):
    kmeans_validator = KMeans(n_clusters=k_candidate, random_state=42, n_init=10)
    validation_labels = kmeans_validator.fit_predict(normalized_embeddings)
    score = silhouette_score(normalized_embeddings, validation_labels)
    silhouette_metrics[k_candidate] = score
    print(f" Candidate Clusters (k) = {k_candidate}, Silhouette Score = {score:.4f}")

# K with highest silhouette score
hss_k = max(silhouette_metrics, key=silhouette_metrics.get)
print(f"\nHighest silhouette score k = {hss_k} (We've chosen not to use this for choosing K)")

# Run final clustering execution pass
cluster_model = KMeans(n_clusters=5, random_state=42, n_init=10)
df["cluster_id"] = cluster_model.fit_predict(normalized_embeddings)

# Project high-dimensional latent space to 2D using UMAP
umap_reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
projected_coordinates = umap_reducer.fit_transform(normalized_embeddings)
df["umap_x"] = projected_coordinates[:, 0]
df["umap_y"] = projected_coordinates[:, 1]

print("\nEngine processing complete. Group distributions calculated.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Compiling and embedding multi-layered text features (Title + Abstract + Keywords)
Evaluating cluster separation quality across parametric range (k=3 to k=10)
 Candidate Clusters (k) = 3, Silhouette Score = 0.0607
 Candidate Clusters (k) = 4, Silhouette Score = 0.0599
 Candidate Clusters (k) = 5, Silhouette Score = 0.0412
 Candidate Clusters (k) = 6, Silhouette Score = 0.0606
 Candidate Clusters (k) = 7, Silhouette Score = 0.0485
 Candidate Clusters (k) = 8, Silhouette Score = 0.0393
 Candidate Clusters (k) = 9, Silhouette Score = 0.0542
 Candidate Clusters (k) = 10, Silhouette Score = 0.0477

Highest silhouette score k = 3 (We've chosen not to use this for choosing K)

Engine processing complete. Group distributions calculated.


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



In [ ]:
# Extract most central thesis records per cluster
for cluster_idx in sorted(df["cluster_id"].unique()):
    cluster_center = final_cluster_engine.cluster_centers_[cluster_idx]
    members_mask = df["cluster_id"] == cluster_idx

    members_embeddings = normalized_embeddings[members_mask]
    member_records = df[members_mask].copy()

    # Compute Euclidean vector distance to cluster center
    vector_distances = np.linalg.norm(members_embeddings - cluster_center, axis=1)
    member_records["distance_to_centroid"] = vector_distances

    print(f"Central Cluster Signatures: ID {cluster_idx} (n={len(member_records)})")
    top_records = member_records.nsmallest(5, "distance_to_centroid")
    for rank, (_, row) in enumerate(top_records.iterrows(), 1):
        print(f" Rank {rank} | Distance: {row['distance_to_centroid']:.4f} | Title: {row['title']}")

Central Cluster Signatures: ID 0 (n=10)
 Rank 1 | Distance: 0.6998 | Title: Image classification using machine learning methods
 Rank 2 | Distance: 0.7207 | Title: An Empirical Comparison of Text Augmentation Techniques in Neural Text Classification
 Rank 3 | Distance: 0.7282 | Title: USING SYNTHETIC TEXT DATA TO  TRAIN SENTIMENT ANALYSIS MODELS
 Rank 4 | Distance: 0.7418 | Title: Using Multimodal Large Language Models for Photo Analysis
 Rank 5 | Distance: 0.7448 | Title: Graph Neural Networks for Text Classification: An Empirical Comparison with Recurrent Networks
Central Cluster Signatures: ID 1 (n=11)
 Rank 1 | Distance: 0.6792 | Title: Innovating Large Language Models (LLMs) with Fine-Tuned Retrieval-Augmented Generation (RAG): Design and Evaluation
 Rank 2 | Distance: 0.7415 | Title: Evaluation of AI tools as a financial risk mitigator: A case study on Emplifi
 Rank 3 | Distance: 0.7457 | Title: Comparing Large Language Models: Analysis of Functionality and Limitations
 Rank 4 | 

In [ ]:
# Stability check
base = KMeans(n_clusters=5, random_state=42, n_init=10).fit_predict(normalized_embeddings)
aris = [adjusted_rand_score(base,
        KMeans(n_clusters=5, random_state=s, n_init=10).fit_predict(normalized_embeddings))
        for s in range(1, 21)]
print(f"Stability across 20 seeds with mean ARI: {np.mean(aris):.3f} (min {np.min(aris):.3f})")

Stability across 20 seeds with mean ARI: 0.461 (min 0.337)


In [ ]:
# c-TF-IDF cluster terms
cluster_docs = (df.assign(txt=combined_text_series)
                  .groupby("cluster_id")["txt"].apply(" ".join))
vec = TfidfVectorizer(stop_words="english", max_features=2000, ngram_range=(1, 2))
M = vec.fit_transform(cluster_docs)
terms = np.array(vec.get_feature_names_out())
for cid, row in zip(cluster_docs.index, M.toarray()):
    print(f"Cluster {cid}: {', '.join(terms[row.argsort()[::-1][:8]])}")

Cluster 0: classification, data, augmentation, networks, neural, model, models, class
Cluster 1: models, ai, language, language models, thesis, llm, large, large language
Cluster 2: misinformation, fact, language, argumentation, cloud, check, fact check, data
Cluster 3: chatbot, thesis, chatgpt, implementation, language, python, large, web
Cluster 4: analysis, sentiment, sentiment analysis, thesis, social, youtube, social networks, corpora


In [ ]:
# Dictionary mapping backed by empirical centroid text signatures
CLUSTER_LABELS = {
    0: "Applied ML & Multimodal Systems",
    1: "LLM Systems: Evaluation, Governance & Limitations",
    2: "Misinformation & Content Integrity",
    3: "Conversational AI & Chatbot Applications",
    4: "Sentiment Analysis & Social Text Analytics",
}

# Apply mapping to dataframe structure
df["cluster_label"] = df["cluster_id"].map(CLUSTER_LABELS)

# Calculate UMAP centroids for each cluster to visualize the 'heart' of each topic
centroids = df.groupby("cluster_label")[["umap_x", "umap_y"]].mean().reset_index()

# Generate interactive scatter plot mapping UMAP dimensions
fig_thematic = px.scatter(
    df,
    x="umap_x",
    y="umap_y",
    color="cluster_label",
    hover_name="title",
    hover_data={
        "author": True,
        "year": True,
        "program": True,
        "faculty": True,
        "umap_x": False,
        "umap_y": False,
        "cluster_id": False
    },
    title="Thematic Map of AI/ML Research Boundaries at VŠE (UMAP Projection)",
    labels={"cluster_label": "Research Pillar"},
    color_discrete_sequence=px.colors.qualitative.Bold,
    template="plotly_white"
)

# Add Centroids as a separate layer
fig_thematic.add_trace(
    go.Scatter(
        x=centroids["umap_x"],
        y=centroids["umap_y"],
        mode="markers+text",
        marker=dict(symbol="diamond", size=14, color="grey", line=dict(width=2, color="white")),
        text=centroids["cluster_label"],
        textposition="top center",
        name="Cluster Centroids",
        hoverinfo="text"
    )
)

# Optimize trace markers and canvas alignment
fig_thematic.update_traces(marker=dict(size=9, opacity=0.85, line=dict(width=1, color="DarkSlateGrey")), selector=dict(mode="markers"))
fig_thematic.update_layout(
    height=600,
    legend=dict(orientation="h", yanchor="bottom", y=-0.3, xanchor="center", x=0.5),
    xaxis=dict(showgrid=True, zeroline=False, showticklabels=False, title="Semantic Space across Axis X"),
    yaxis=dict(showgrid=True, zeroline=False, showticklabels=False, title="Semantic Space across Axis Y")
)

fig_thematic.show()

### Interpretation

K-means at k=5 partitions the 55 theses into five interpretable clusters:

**Sentiment Analysis & Social Text Analytics (n=19, largest):** The historical backbone of text analytics at FIS. Centroid theses are near-identical sentiment-on-Twitter/social-networks studies plus political text mining. Many theses repeat the same setup and method.

**LLM Systems: Evaluation, Governance & Limitations (n=11):** RAG design, model comparison, LLM moderation, AI misalignment. The most research-leaning cluster: students probe how LLMs behave and where they fail rather than just using them.

**Applied ML & Multimodal Systems (n=10):** Image classification, multimodal photo analysis, text augmentation, synthetic data, GNNs for classification. The most technically experimental cluster — original model work across modalities.

**Misinformation & Content Integrity (n=8):** Fake-news classification, malicious-content detection, source credibility, fact-checking with LLMs. Small but unusually coherent and self-contained, with clear external relevance (media, platform trust, public sector).

**Conversational AI & Chatbot Applications (n=7):** Chatbots and ChatGPT-based interfaces for concrete tasks (scout org, invoicing, information search). The most practitioner-oriented cluster: deploying conversational tooling rather than studying it.

## View: Study Program Profile

**Question:** How are AI/ML research topics distributed across study programs?

**Relevance:** Curriculum decisions are program-specific — knowing *which* program concentrates on which cluster tells the Vice-Dean where a new course actually belongs.

**Method:** Cross-tabulation of the `program` column against k=5 cluster labels. Programs with fewer than 5 theses are grouped as "Other." Row-normalized percentages keep smaller programs comparable to Applied Informatics.

In [ ]:


# Keep only programs with >= 5 theses
prog_counts = df["program"].value_counts()
df.loc[df["program"].isin(prog_counts[prog_counts < 5].index), "program"] = "Other"

# Cross-tabulate: programs*clusters
ct = pd.crosstab(df["program"], df["cluster_label"])
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100   # row-normalised

# Heatmap
fig = go.Figure(go.Heatmap(
    z=ct_pct.values,
    x=ct_pct.columns.tolist(),
    y=ct_pct.index.tolist(),
    colorscale="Blues",
    text=ct_pct.round(0).astype(int).astype(str) + "%",
    texttemplate="%{text}",
    hovertemplate="Program: %{y}<br>Cluster: %{x}<br>Share: %{z:.1f}%<extra></extra>",
))
fig.update_layout(
    title="Topic Cluster Distribution by Study Program (% of program's theses)",
    xaxis_title="Topic cluster",
    yaxis_title="Study program",
    height=400,
)
fig.show()

### Interpretation

**Applied Informatics (n=30)** is broadly distributed across all five clusters, which makes sense as the largest and most technically varied program. The slight lean toward Sentiment Analysis (27%) is a legacy effect — it's the longest-running template. The meaningful signal is that 23% sit in LLM Governance and 20% in Conversational AI, confirming this program is already absorbing the generative AI shift. New course offerings land here first.

**Data Analytics (n=10)** is the most distinctive profile: 50% in Applied ML & Multimodal, 30% in Misinformation & Content Integrity, zero in Conversational AI. These students gravitate toward structured, experimental work — classification pipelines, augmentation experiments, data-driven fact-checking — rather than deploying chat interfaces. Curriculum investment here should prioritize ML tooling and evaluation methodology, not another LLM applications course.

**Other programs (n=15, mostly business/management)** are dominated by Sentiment Analysis at 67%, with zero Applied ML. This reflects the accessibility of sentiment analysis as a methodology for non-CS students: minimal ML infrastructure, maps cleanly onto business questions about brand perception or public opinion. No technical expansion is realistic here without a stronger quantitative prerequisite.

## View: Application Domains

**Question:** Which real-world industries and application domains appear most frequently across theses?

**Relevance:** Industry partnerships require knowing where FIS student work already overlaps with business needs, a domain appearing in many theses signals existing supervisor expertise and available datasets that FIS can leverage for internship or sponsored-project agreements.

**Method:** Regex-based domain classifier applied to each abstract + title. The domain taxonomy was built bottom-up by reading ~30 abstracts and identifying recurring application areas. Each domain has a primary and optional secondary pattern to reduce false negatives. Multi-label: a thesis matching two domains is counted in both. Preferred over an LLM classifier here because the vocabulary is consistent and technical, and the rules are fully transparent and inspectable.

In [ ]:
DOMAIN_PATTERNS = {
    "Finance & Trading":      r'\b(financ|trading|stock|forex|insurance|bank|investment|portfolio|credit|fraud)\b',
    "Healthcare & Medicine":  r'\b(medical|health|clinical|patient|diagnos|disease|cancer|hospital|drug|pharma)\b',
    "E-commerce & Retail":    r'\b(e-commerce|ecommerce|retail|product|recommend|shop|customer|pricing|inventory)\b',
    "Cybersecurity":          r'\b(cyber|security|malware|intrusion|attack|threat|vulnerab|phishing|anomaly detection)\b',
    "NLP & Information":      r'\b(news|article|sentiment|review|opinion|text classif|search|document|chatbot|information retrieval)\b',
    "Education & HR":         r'\b(education|learning|student|academic|HR|recruitment|employee|talent)\b',
    "Sports & Entertainment": r'\b(sport|football|basketball|music|game|video|film|esport)\b',
    "Social Media & Marketing": r'\b(social media|twitter|instagram|marketing|advertis|brand|influencer)\b',
    "Transport & Logistics":  r'\b(transport|logistics|traffi|route|delivery|supply chain|autonomous)\b',
    "Energy & Environment":   r'\b(energy|climate|environment|emission|solar|wind|smart grid)\b',
}

records = []
for _, row in df_raw.iterrows():
    text = (str(row["abstract"]) + " " + str(row["title"])).lower()
    matched = False
    for domain, pattern in DOMAIN_PATTERNS.items():
        if re.search(pattern, text, re.IGNORECASE):
            records.append({"url": row["url"], "domain": domain,
                            "title": row["title"], "cluster_label": row.get("cluster_label", "")})
            matched = True
    if not matched:
        records.append({"url": row["url"], "domain": "General / Methodology",
                        "title": row["title"], "cluster_label": row.get("cluster_label", "")})

domain_df = pd.DataFrame(records)

# Thesis-level count (a thesis with 2 domains is counted in each)
domain_counts = domain_df.groupby("domain")["url"].nunique().sort_values(ascending=True)
fig = px.bar(
    x=domain_counts.values,
    y=domain_counts.index,
    orientation="h",
    title="Application Domains in AI/ML Bachelor Theses",
    labels={"x": "Number of theses", "y": "Domain"},
    color=domain_counts.values,
    color_continuous_scale="Teal",
)
fig.update_layout(coloraxis_showscale=False, height=500)

print(f"\nTotal domain assignments: {len(domain_df)}")
print(f"Theses matching at least one domain: {domain_df['url'].nunique()}")
fig.show()


Total domain assignments: 93
Theses matching at least one domain: 55


###Interpretation

**NLP & Information (n=31)** is the dominant domain by a wide margin — but this is partly a corpus artifact. The four search queries (NLP, sentiment analysis, transformer, LLM) systematically pull in text-processing work, so the count confirms selection criteria as much as genuine domain concentration. Read it as a lower bound on FIS's NLP strength, not an unbiased signal.

**Education & HR (n=19)** warrants caution before treating it as a partnership signal. The pattern catches words like "student," "learning," and "academic" that appear routinely in thesis prose in a methodological context, not because the thesis is actually about education technology. Spot-checking a few matches is advisable before presenting this to the Vice-Dean.

**Social Media & Marketing (n=12)** is clean and expected — it maps directly onto the Sentiment Analysis cluster and reflects genuine student interest in brand/opinion monitoring. A credible industry signal for marketing analytics partnerships.

**Healthcare (n=3) and Finance (n=3)** are the meaningful absences. Both are among the highest-demand ML application areas in industry, and both are nearly absent from student work. Given that FIS sits within an economics university, Finance in particular is a notable gap — the corpus contains almost no work on forecasting, risk, or trading despite an obvious institutional fit.

**Energy & Environment (n=9)** is plausible but worth a quick manual check — the regex pattern is broad enough to pick up false positives. If confirmed, it would be a minor surprise given the corpus selection criteria.

## View:  Technology & Tools Landscape

**Question:** What specific models, frameworks, libraries, and tools do students mention?

**Relevance:** The most directly actionable view for curriculum planning — tool frequency tells the Vice-Dean what to teach, and the trend over time signals what to update.

**Method:** Gazetteer-based entity extraction over `abstract + keywords`. The technology list was compiled from domain knowledge and extended by scanning the top 50 corpus unigrams/bigrams. Terms are grouped into categories (ML Library, LLM, Architecture, etc.) for the bar chart. The `keywords` column carries particular weight here,, students often list tools directly that don't appear in the abstract prose. A co-occurrence network is added: if two tools appear in the same thesis an edge is drawn, revealing which combinations are common in practice.

In [ ]:
TECH_DICT = {
    # ML frameworks and libraries
    "scikit-learn":   r'\bscikit-?learn\b',
    "PyTorch":        r'\bpytorch\b',
    "TensorFlow":     r'\btensorflow\b',
    "Keras":          r'\bkeras\b',
    "XGBoost":        r'\bxgboost\b',
    "LightGBM":       r'\blightgbm\b',
    "Random Forest":  r'\brandom forest\b',
    "SVM":            r'\b(svm|support vector)\b',
    # LLMs and foundation models
    "GPT / ChatGPT":  r'\b(gpt-?[234]?o?|chatgpt|openai)\b',
    "BERT":           r'\bbert\b',
    "LLaMA":          r'\bllama\b',
    "Gemini":         r'\bgemini\b',
    "Claude":         r'\bclaude\b',
    "RAG":            r'\b(rag|retrieval.augmented)\b',
    # Deep learning architectures
    "CNN":            r'\b(cnn|convolutional neural)\b',
    "LSTM":           r'\blstm\b',
    "Transformer":    r'\btransformer\b',
    "Autoencoder":    r'\bautoencoder\b',
    # Data anmd platforms
    "Python":         r'\bpython\b',
    "R":              r'\b(r language|\bin r\b|rstudio)\b',
    "Pandas":         r'\bpandas\b',
    "Hugging Face":   r'\bhugging.?face\b',
    "LangChain":      r'\blangchain\b',
    "spaCy":          r'\bspacy\b',
}

TECH_CATEGORIES = {
    "scikit-learn": "ML Library", "PyTorch": "ML Framework", "TensorFlow": "ML Framework",
    "Keras": "ML Framework", "XGBoost": "ML Library", "LightGBM": "ML Library",
    "Random Forest": "Algorithm", "SVM": "Algorithm",
    "GPT / ChatGPT": "LLM", "BERT": "LLM", "LLaMA": "LLM",
    "Gemini": "LLM", "Claude": "LLM", "RAG": "LLM Pattern",
    "CNN": "Architecture", "LSTM": "Architecture", "Transformer": "Architecture",
    "Autoencoder": "Architecture",
    "Python": "Language", "R": "Language", "Pandas": "Library",
    "Hugging Face": "Platform", "LangChain": "Platform", "spaCy": "NLP Library",
}

# Scan both abstract and keywords
df["search_text"] = (df["abstract"].fillna("") + " " + df["keywords"].fillna("")).str.lower()

tech_counts = Counter()
thesis_tech = defaultdict(list)

for _, row in df.iterrows():
    text = row["search_text"]
    for tool, pattern in TECH_DICT.items():
        if re.search(pattern, text, re.IGNORECASE):
            tech_counts[tool] += 1
            thesis_tech[row["url"]].append(tool)

tech_df = pd.DataFrame([
    {"tool": t, "count": c, "category": TECH_CATEGORIES.get(t, "Other")}
    for t, c in tech_counts.most_common()
])

fig = px.bar(
    tech_df.sort_values("count"),
    x="count", y="tool", orientation="h", color="category",
    title="Technology & Tool Mentions Across Thesis Abstracts + Keywords",
    labels={"count": "Number of theses", "tool": ""},
    color_discrete_sequence=px.colors.qualitative.Pastel,
    height=600,
)
fig.show()

# --- Co-occurrence network (tools appearing together in ≥3 theses) ---
G = nx.Graph()
for tools in thesis_tech.values():
    for i, t1 in enumerate(tools):
        for t2 in tools[i+1:]:
            if G.has_edge(t1, t2):
                G[t1][t2]["weight"] += 1
            else:
                G.add_edge(t1, t2, weight=1)

# Keep edges with weight >= 2
edges_to_remove = [(u, v) for u, v, d in G.edges(data=True) if d["weight"] < 2]
G.remove_edges_from(edges_to_remove)
isolated = list(nx.isolates(G))
G.remove_nodes_from(isolated)

pos = nx.spring_layout(G, seed=42, k=2)
edge_x, edge_y = [], []
for u, v in G.edges():
    x0, y0 = pos[u]; x1, y1 = pos[v]
    edge_x += [x0, x1, None]; edge_y += [y0, y1, None]

node_x = [pos[n][0] for n in G.nodes()]
node_y = [pos[n][1] for n in G.nodes()]
node_text = list(G.nodes())
node_size = [tech_counts[n] * 2 for n in G.nodes()]

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=edge_x, y=edge_y, mode="lines",
                          line=dict(width=0.8, color="#aaa"), hoverinfo="none"))
fig2.add_trace(go.Scatter(x=node_x, y=node_y, mode="markers+text",
                          text=node_text, textposition="top center",
                          marker=dict(size=node_size, color="steelblue", opacity=0.8),
                          hoverinfo="text"))
fig2.update_layout(title="Tool Co-occurrence Network (edges = appear together in ≥3 theses)",
                   showlegend=False, xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                   yaxis=dict(showgrid=False, zeroline=False, showticklabels=False), height=500)
fig2.show()

print(f"\nTop 15 tools:\n{tech_df.sort_values('count', ascending=False).head(15).to_string(index=False)}")


Top 15 tools:
         tool  count     category
GPT / ChatGPT     10          LLM
       Python      6     Language
  Transformer      6 Architecture
          CNN      4 Architecture
Random Forest      3    Algorithm
       Gemini      3          LLM
          RAG      3  LLM Pattern
       Claude      3          LLM
         LSTM      3 Architecture
      XGBoost      2   ML Library
         BERT      2          LLM
          SVM      1    Algorithm
    LangChain      1     Platform
        LLaMA      1          LLM


###Interpretation

**GPT / ChatGPT dominates (n=10)** — mentioned in roughly 1 in 5 theses, almost twice the next tool. Combined with Gemini (3) and Claude (3), OpenAI-ecosystem and frontier LLMs account for the majority of named model references. This is a deployment pattern: students reach for the most accessible, best-marketed tool rather than building with open-weight alternatives.

**Classic ML frameworks are nearly absent.** scikit-learn, PyTorch, TensorFlow, Keras, and Hugging Face don't appear in the top 15 at all. Two explanations: abstracts rarely name implementation libraries unless they're the thesis topic, or students genuinely aren't building with them — using APIs instead. Either way it's a curriculum signal: if the goal is producing practitioners who can actually train and evaluate models, framework coverage needs to be explicit, not assumed.

**Architecture vocabulary (Transformer n=6, CNN n=4, LSTM n=3) appears without matching framework mentions.** Students describe *what* they used conceptually but don't name *how* they implemented it.

**RAG (n=3) and LangChain (n=1)** are early signals of the agentic/retrieval pattern entering student work. Low now but a reasonable bet to grow quickly given the LLM Governance cluster's size.

**A co-occurrence network** was attempted but yielded only 4 nodes most theses name at most one tool explicitly in the abstract, making co-occurrence analysis uninformative at this corpus size.

## View: Research Depth Classification

**Question:** What is the balance between theses developing original models vs. deploying existing tools?

**Relevance:** Shapes FIS's positioning: "AI researchers" vs. "AI practitioners" are both legitimate but imply different course offerings, thesis supervision styles, and industry partnerships.

**Method:** Rule-based signal classifier applied to each abstract. Intentionally simple and transparent — abstract-level classification cannot reliably distinguish research depth with high confidence, so a rule system is more honest than a trained classifier that would overstate precision. Signals were derived by manually reading 30 abstracts and listing the verbs distinguishing original work from deployment. Three categories: **Original / Experimental**, **Tool Deployment**, **Mixed**. The output is treated as an estimate; a random sample should be spot-checked to gauge real precision.

In [ ]:
ORIGINAL_SIGNALS = re.compile(
    r'\b(propos(e|es|ed|ing)|novel|new (model|approach|method|architecture|framework)|'
    r'design(s|ed|ing) (a|an|our)|develop(s|ed|ing) (a|an|our)|implement(s|ed|ing) (a|an|our)|'
    r'our (model|approach|method|system)|we (train|propose|introduce|present)|'
    r'from scratch|custom (model|architecture)|original (dataset|model)|'
    r'experiment(al|s|ed|ing)|evaluat(e|es|ed|ing) (our|the proposed))\b',
    re.IGNORECASE
)

DEPLOYMENT_SIGNALS = re.compile(
    r'\b(appl(y|ies|ied|ication) (of |existing |a )(tool|framework|model|method)|'
    r'us(e|es|ed|ing) (existing|available|pre-?trained|off-the-shelf)|'
    r'compar(e|es|ed|ing) (existing|available|state-of-the-art)|'
    r'evaluat(e|es|ed|ing) (existing|available|commercial)|'
    r'deploy(s|ed|ing|ment)|leverag(e|es|ed|ing)|'
    r'integrat(e|es|ed|ing) (an? )?(existing|available)|'
    r'(chatgpt|gpt-4|llama|gemini|claude) (to|for|was used))\b',
    re.IGNORECASE
)

def classify_depth(abstract):
    text = str(abstract)
    orig = bool(ORIGINAL_SIGNALS.search(text))
    depl = bool(DEPLOYMENT_SIGNALS.search(text))
    if orig and not depl:
        return "Original / Experimental"
    elif depl and not orig:
        return "Tool Deployment"
    elif orig and depl:
        return "Mixed"
    else:
        return "Unclear"

df["research_depth"] = df["abstract"].apply(classify_depth)

counts = df["research_depth"].value_counts()
print(counts)

counts_classified = counts[counts.index != "Unclear"]
fig = px.pie(
    values=counts_classified.values,
    names=counts_classified.index,
    title="Research Depth: Original Work vs. Tool Deployment (classified theses only)",
    color_discrete_sequence=px.colors.qualitative.Set2,
    hole=0.35,
)
fig.update_traces(textposition="outside", textinfo="percent+label")
fig.show()
pct_unclear = (counts.get("Unclear", 0) / counts.sum() * 100)
print(f"Note: {pct_unclear:.0f}% of theses unclassifiable and were excluded from chart")

# Cross-tab with cluster (requires View 1 to have run)
if "cluster_label" in df.columns:
    ct = pd.crosstab(df["cluster_label"], df["research_depth"], normalize="index") * 100
    print("\nResearch depth by topic cluster (%):")
    print(ct.round(1))

research_depth
Unclear                    34
Original / Experimental    17
Mixed                       2
Tool Deployment             2
Name: count, dtype: int64


Note: 62% of theses unclassifiable and were excluded from chart

Research depth by topic cluster (%):
research_depth                                     Mixed  \
cluster_label                                              
Applied ML & Multimodal Systems                      0.0   
Conversational AI & Chatbot Applications             0.0   
LLM Systems: Evaluation, Governance & Limitations   18.2   
Misinformation & Content Integrity                   0.0   
Sentiment Analysis & Social Text Analytics           0.0   

research_depth                                     Original / Experimental  \
cluster_label                                                                
Applied ML & Multimodal Systems                                       50.0   
Conversational AI & Chatbot Applications                              28.6   
LLM Systems: Evaluation, Governance & Limitations                     18.2   
Misinformation & Content Integrity                                    62.5   
Sentiment

### Interpretation

**Classifiability itself varies by cluster:** Applied ML (50%) and Misinformation (62.5%) have the highest share of classifiable abstracts, while Conversational AI (28.6%) and Sentiment Analysis (21%) are mostly unclear. This tracks: experimental work tends to use methodology-signaling language; tool-deployment and template-style theses describe outcomes instead.

**Among classified theses:** Applied ML and Misinformation lean strongly Original (100% and 100% of their classified theses respectively). LLM Systems is the only cluster with Mixed signal (18%) and has the largest Tool Deployment share (9%), with Sentiment Analysis a distant second (5%)., students in that cluster are more likely to combine evaluation of existing models with their own experiments. Sentiment Analysis's small classified share skews Original, but the base is too small to read confidently.

**Limitation:** The classifier captures explicit methodological language, it systematically misses deployment work where students describe results rather than process. "Unclear" should be read as "abstract doesn't signal contribution type," not "no contribution."

## View: Supervisor Expertise Network

**Question:** Which supervisors are driving AI/ML thesis production at FIS, and do they concentrate around specific topics?

**Relevance:** Any new specialisation requires supervising capacity, not just student demand. This view answers: *do we have the faculty to staff a new track?* It also surfaces single-points-of-failure — topics entirely dependent on one supervisor.

**Method:** A bipartite network connects supervisors to the topic clusters of their supervised theses (edge weight = thesis count). The network is then projected onto supervisors only — two supervisors share an edge if they supervise theses in the same cluster, revealing expertise overlap. Node size encodes total thesis count. Community detection (`greedy_modularity_communities`) groups supervisors into expertise clusters. Supervisors with fewer than 2 theses in the corpus are excluded to keep the graph readable.

In [ ]:
# Clean supervisor names (remove trailing whitespace, etc.)
df["supervisor_clean"] = df["supervisor"].str.strip()

# Keep supervisors with >= 2 theses in corpus
sup_counts = df["supervisor_clean"].value_counts()
active_supervisors = sup_counts[sup_counts >= 2].index
df_sup = df[df["supervisor_clean"].isin(active_supervisors)].copy()

print(f"Active supervisors (≥2 theses): {len(active_supervisors)}")
print(f"Theses covered: {len(df_sup)} / {len(df)}")

# --- Build bipartite graph: supervisor ↔ cluster ---
B = nx.Graph()
for _, row in df_sup.iterrows():
    sup = row["supervisor_clean"]
    cluster = row["cluster_label"]
    if B.has_edge(sup, cluster):
        B[sup][cluster]["weight"] += 1
    else:
        B.add_edge(sup, cluster, weight=1)

# --- Supervisor-only projection ---
supervisors = set(df_sup["supervisor_clean"].unique())
G_sup = nx.Graph()
for cluster in df_sup["cluster_label"].unique():
    cluster_supervisors = [
        n for n in B.neighbors(cluster) if n in supervisors
    ]
    for i, s1 in enumerate(cluster_supervisors):
        for s2 in cluster_supervisors[i+1:]:
            if G_sup.has_edge(s1, s2):
                G_sup[s1][s2]["weight"] += 1
            else:
                G_sup.add_edge(s1, s2, weight=1)

# Add isolated supervisors (supervise only one cluster)
for sup in supervisors:
    if sup not in G_sup:
        G_sup.add_node(sup)

# Community detection
communities = list(greedy_modularity_communities(G_sup))
community_map = {}
for i, comm in enumerate(communities):
    for node in comm:
        community_map[node] = i

# Layout
pos = nx.spring_layout(G_sup, seed=42, k=3, weight="weight")

# Build Plotly traces
edge_x, edge_y = [], []
for u, v in G_sup.edges():
    x0, y0 = pos[u]; x1, y1 = pos[v]
    edge_x += [x0, x1, None]; edge_y += [y0, y1, None]

node_x = [pos[n][0] for n in G_sup.nodes()]
node_y = [pos[n][1] for n in G_sup.nodes()]
node_labels = list(G_sup.nodes())
node_sizes = [sup_counts.get(n, 1) * 6 for n in G_sup.nodes()]
node_colors = [community_map.get(n, 0) for n in G_sup.nodes()]
node_hover = [
    f"{n}<br>Theses: {sup_counts.get(n, 0)}<br>"
    f"Clusters: {', '.join(df_sup[df_sup['supervisor_clean']==n]['cluster_label'].unique())}"
    for n in G_sup.nodes()
]

fig = go.Figure()
fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode="lines",
                         line=dict(width=0.6, color="#ccc"), hoverinfo="none"))
fig.add_trace(go.Scatter(
    x=node_x, y=node_y, mode="markers+text",
    text=node_labels, textposition="top center",
    marker=dict(size=node_sizes, color=node_colors,
                colorscale="Viridis", showscale=False, opacity=0.85),
    hovertext=node_hover, hoverinfo="text",
))
fig.update_layout(
    title="Supervisor Expertise Network — node size = thesis count, colour = community",
    showlegend=False,
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    height=600,
)
fig.show()

Active supervisors (≥2 theses): 11
Theses covered: 30 / 55


###Interpretation
**Chudán, David supervises the most theses (7) and is the only one with meaningful coverage across all five clusters.** In practice this means a lot of the AI/ML breadth at FIS runs through one person. That's worth flagging — not as a criticism, but as a capacity risk if the faculty wants to scale up.

**Sentiment Analysis has the healthiest supervision coverage:** Jelínek (3), Kincl (2), and Chudán (2) all actively supervise in this area. It's the most resilient cluster from a staffing perspective, which probably contributes to why students keep doing sentiment analysis theses — there are supervisors who know the territory well.

**Several supervisors have clear specialisations:**
- Kliegr focuses on Misinformation & Content Integrity
- Kovářová and Zeman are both concentrated in LLM Systems
- Berka and Zamazal are the go-to people for Applied ML work

**Conversational AI has no natural home supervisor:** it's split between Chudán, Kučera, and Luc with one thesis each. If this area is to grow it needs someone to develop expertise and actively take it on.

#Executive Summary

This report analyzed 55 FIS bachelor theses containing AI/ML content, scraped
from the VŠKP portal using four search terms: natural language processing,
sentiment analysis, transformer, and large language model. All findings below
are grounded in the five analytical views and should be read with one caveat
in mind: the corpus selection biases toward NLP and LLM work, so the results
reflect the text-analytics corner of FIS's AI/ML landscape, not the full picture.

---

###What are students actually working on?

As shown in the Thematic Map (View 1), five research areas emerge from the corpus:

**Sentiment Analysis and Social Text Analytics (n=19)** is the largest and
longest-running track. Well-supervised, well-understood, but saturated. A lot
of these theses are doing the same thing in slightly different contexts.

**LLM Systems: Evaluation, Governance and Limitations (n=11)** is where the
more research-oriented students are ending up. RAG pipelines, model comparisons,
alignment, moderation. The most intellectually interesting cluster right now.

**Applied ML and Multimodal Systems (n=10)** covers image classification,
synthetic data, text augmentation, GNNs. The most experimentally rigorous
cluster and currently the most underrepresented relative to its potential.

**Misinformation and Content Integrity (n=8)** is small but unusually coherent.
Fake news detection, source credibility, fact-checking with LLMs. Clear societal
relevance and a credible niche.

**Conversational AI and Chatbot Applications (n=7)** covers practical chatbot
and interface deployments. The most practitioner-oriented area in the corpus.

---

### Curriculum

The Study Program Profile (View 2) shows that Applied Informatics students are
already gravitating toward LLM Systems (23%) and Conversational AI (20%) on
their own. A dedicated module on LLM evaluation, fine-tuning, and responsible
deployment would formalize what is already happening organically rather than
introducing something new.

Data Analytics tells a different story: 50% of its theses fall into Applied ML,
which means these students want structured, experimental, quantitative work.
More NLP courses are not what this program needs. ML tooling depth, evaluation
methodology, and framework usage are.

The Technology Landscape (View 4) makes the framework gap explicit. GPT/ChatGPT
is the most mentioned tool by a significant margin. scikit-learn, PyTorch, and
Hugging Face barely appear. Students are using APIs rather than building with ML
frameworks. If graduating students who can actually train and evaluate models is
a goal, framework usage needs to be an explicit course requirement somewhere,
not assumed background knowledge.

Sentiment Analysis is well-covered and well-supervised. It does not need more
investment. Redirecting some of that supervision capacity toward LLM Systems or
Conversational AI would have more strategic value.

---

###Student recruitment and marketing

FIS can credibly market LLM application skills. The theses show genuine
engagement with generative AI, RAG systems, and conversational interfaces.
What FIS cannot yet credibly market is model-building depth, because the
experimental and framework signals are thin (Views 4 and 5).

The honest positioning is: FIS produces AI practitioners who can apply and
evaluate state-of-the-art tools, with a growing cohort doing more serious
LLM-oriented work. The Misinformation and Content Integrity cluster adds a
credible ethics and media angle that tends to resonate with prospective students.

---

### Industry partnerships

The Application Domains view (View 3) gives a clearer picture here than the
thematic clusters. Social media and marketing analytics is the strongest
immediate target: the student ecosystem, the supervisor expertise, and the
available datasets are all already in place. Media monitoring companies,
marketing agencies, and PR firms are the natural first calls.

Healthcare and finance are the notable gaps. Both are high-demand ML domains
in industry and both appear in fewer than 5 theses. Finance is particularly
worth flagging given FIS's position within an economics university. The
institutional fit is obvious and the gap is addressable.

Misinformation and Content Integrity is worth approaching selectively.
Fact-checking organizations, public broadcasters, and platform trust and
safety teams are realistic partners for sponsored thesis projects in this area.

---

### And for last
The Supervisor Expertise Network (View 6) shows that a significant share of
AI/ML thesis supervision at FIS is concentrated in a very small number of
people, with one supervisor covering more ground than anyone else by a
noticeable margin. The clusters with the most growth potential, LLM Systems,
Applied ML, and Conversational AI, also have the thinnest and most fragmented
supervision coverage.

Any of the curriculum or partnership recommendations above are only realistic
if the supervision capacity question is addressed alongside them. Designing
new courses or signing industry agreements without a concrete plan for who
supervises the resulting theses is optimistic at best.